# 🚀 Exemplo: Função Geral de Treinamento PatchTST

Este notebook demonstra como usar a função `train_and_evaluate_patchtst()` para:
- Treinar um modelo PatchTST com configurações personalizadas
- Computar métricas automaticamente
- Avaliar em todos os conjuntos (treino, validação, teste)
- Visualizar previsões vs valores reais
- Comparar múltiplas configurações facilmente

In [4]:
# 1. Importar bibliotecas
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.append('../src')

from dataset import RepositorioDados
from models.transformer import train_and_evaluate_patchtst

print("✓ Bibliotecas importadas")

✓ Bibliotecas importadas


In [5]:
# 2. Configurar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32

print(f"Device: {device} | Precision: {dtype}")

Device: cpu | Precision: torch.float32


In [6]:
# 3. Preparar dados
TIMESTAMP_COLUMN = 'timestamp'
TARGET_COLUMN = ['Vol']
FEATURES = ["Vol_lag_1", "Vol_week_mean", "Vol_month_mean"]
ID_COLUMNS = []

CONTEXT_LENGTH = 256
FORECAST_HORIZON = 1
TRAIN_FRAC, VALID_FRAC = 0.7, 0.1

print("Carregando dados...")
repo = RepositorioDados()
tsp, train_ds, valid_ds, test_ds, test_df = repo.executar(
    timestamp_col=TIMESTAMP_COLUMN,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    context_length=CONTEXT_LENGTH,
    features=FEATURES,
    target=TARGET_COLUMN,
    id_cols=ID_COLUMNS,
    forecast_horizon=FORECAST_HORIZON,
    use_mean_features=True,
    lags=1
)

print(f"✓ Dados carregados")
print(f"  Train: {len(train_ds)} | Valid: {len(valid_ds)} | Test: {len(test_ds)}")

Carregando dados...
Carregando dados de C:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade
Treino: 1534 amostras | Val: 256 | Teste: 512
✓ Dados carregados
  Train: 1534 | Valid: 256 | Test: 512


In [7]:
test_df = pd.read_csv('../data/interim/df_vol.csv')
test_df.rename(columns={'date': TIMESTAMP_COLUMN}, inplace=True)
test_df[TIMESTAMP_COLUMN] = pd.to_datetime(test_df[TIMESTAMP_COLUMN])

print(f"✓ test_df carregado: {len(test_df)} linhas")

✓ test_df carregado: 2558 linhas


## 📌 Exemplo 1: Treinamento Básico com Configuração Padrão

In [8]:
"""
🏆 MELHORES HIPERPARÂMETROS (Optuna):
================================================================================
features_idx             : 1
context_length           : 256
forecast_horizon         : 1
d_model                  : 128
num_attention_heads      : 8
num_hidden_layers        : 2
ffn_dim                  : 256
dropout                  : 0.1
patch_length             : 16
learning_rate            : 0.0005
batch_size               : 32
================================================================================
"""

'\n🏆 MELHORES HIPERPARÂMETROS (Optuna):\n================================================================================\nfeatures_idx             : 1\ncontext_length           : 256\nforecast_horizon         : 1\nd_model                  : 128\nnum_attention_heads      : 8\nnum_hidden_layers        : 2\nffn_dim                  : 256\ndropout                  : 0.1\npatch_length             : 16\nlearning_rate            : 0.0005\nbatch_size               : 32\n================================================================================\n'

In [9]:
# Configurar hiperparâmetros
model_config = {
    'do_mask_input': False,
    'context_length': CONTEXT_LENGTH,
    'patch_length': 1,
    'num_input_channels': len(TARGET_COLUMN),
    'prediction_length': FORECAST_HORIZON,
    'd_model': 128,
    'num_attention_heads': 8,
    'num_hidden_layers': 2,
    'ffn_dim': 256,
    'dropout': 0.1,
    'head_dropout': 0.1,
    'channel_attention': True,
    'scaling': 'std',
    'loss': 'mse',
    'pre_norm': True,
    'norm_type': 'batchnorm',
}

training_config = {
    'learning_rate': 0.0005,
    'num_epochs': 50,
    'batch_size': 32,
    'num_workers': 0,
    'early_stopping_patience': 5,
    'early_stopping_threshold': 0.001,
}

print("✓ Configurações definidas")

✓ Configurações definidas


In [10]:
# Executar treinamento com a função geral
# NOTA: Isso é UMA LINHA para treinar, avaliar e visualizar!
results = train_and_evaluate_patchtst(
    train_dataset=train_ds,
    valid_dataset=valid_ds,
    test_dataset=test_ds,
    test_df=test_df,
    tsp=tsp,
    model_config=model_config,
    training_config=training_config,
    device=device,
    dtype=dtype,
    context_length=CONTEXT_LENGTH,
    timestamp_column=TIMESTAMP_COLUMN,
    model_name="PatchTST Volatilidade",
    output_dir="./models/patchtst_baseline",
    verbose=True
)


🚀 INICIANDO TREINAMENTO DO PatchTST Volatilidade

📊 Configuração do Modelo:
   do_mask_input                 : False
   context_length                : 256
   patch_length                  : 1
   num_input_channels            : 1
   prediction_length             : 1
   d_model                       : 128
   num_attention_heads           : 8
   num_hidden_layers             : 2
   ffn_dim                       : 256
   dropout                       : 0.1
   head_dropout                  : 0.1
   channel_attention             : True
   scaling                       : std
   loss                          : mse
   pre_norm                      : True
   norm_type                     : batchnorm

⚙️  Configuração de Treinamento:
   learning_rate                 : 0.0005
   num_epochs                    : 50
   batch_size                    : 32
   num_workers                   : 0
   early_stopping_patience       : 5
   early_stopping_threshold      : 0.001

🎯 Device: cpu | Dtype: torch.fl

TypeError: PatchTSTForPrediction.forward() got an unexpected keyword argument 'future_observed_mask'

In [ ]:
# 5. Acessar resultados
print("\n📊 Métricas do Modelo:")
for metric, value in results['test_metrics'].items():
    print(f"  {metric:10s}: {value:12.6f}")

print(f"\n📈 Previsões vs Reais:")
print(f"  Previsões (primeiras 10): {results['predictions'][:10].round(4)}")
print(f"  Valores Reais (primeiros 10): {results['labels'][:10].round(4)}")

## 🔦 Exemplo 2: Comparar Múltiplas Configurações

In [ ]:
# Definir múltiplas configurações para comparação
configurations = {
    'Pequeno': {
        'model': {
            'd_model': 64,
            'num_attention_heads': 8,
            'num_hidden_layers': 2,
            'ffn_dim': 256,
        },
        'training': {
            'learning_rate': 1e-4,
            'num_epochs': 30,
            'batch_size': 32,
        }
    },
    'Médio': {
        'model': {
            'd_model': 128,
            'num_attention_heads': 16,
            'num_hidden_layers': 3,
            'ffn_dim': 512,
        },
        'training': {
            'learning_rate': 1e-4,
            'num_epochs': 50,
            'batch_size': 32,
        }
    },
    'Grande': {
        'model': {
            'd_model': 256,
            'num_attention_heads': 32,
            'num_hidden_layers': 4,
            'ffn_dim': 1024,
        },
        'training': {
            'learning_rate': 5e-5,
            'num_epochs': 50,
            'batch_size': 16,
        }
    },
}

print(f"🔍 Será realizado teste de {len(configurations)} configurações diferentes")

In [ ]:
# Executar treinamento para cada configuração
all_results = {}

for config_name, config_params in configurations.items():
    print(f"\n{'='*80}")
    print(f"Treinando modelo: {config_name}")
    print(f"{'='*80}")
    
    # Merge com configuração base
    model_cfg = {**model_config, **config_params['model']}
    training_cfg = {**training_config, **config_params['training']}
    
    # Treinar
    results = train_and_evaluate_patchtst(
        train_dataset=train_ds,
        valid_dataset=valid_ds,
        test_dataset=test_ds,
        test_df=test_df,
        tsp=tsp,
        model_config=model_cfg,
        training_config=training_cfg,
        device=device,
        dtype=dtype,
        context_length=CONTEXT_LENGTH,
        timestamp_column=TIMESTAMP_COLUMN,
        model_name=f"PatchTST {config_name}",
        output_dir=f"./models/patchtst_{config_name.lower()}",
        verbose=True
    )
    
    all_results[config_name] = results

In [ ]:
# Criar tabela comparativa
comparison_data = []

for config_name, results in all_results.items():
    metrics = results['test_metrics']
    comparison_data.append({
        'Modelo': config_name,
        'MSE': metrics['MSE'],
        'MAE': metrics['MAE'],
        'RMSE': metrics['RMSE'],
        'MAPE': metrics['MAPE'],
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n📊 COMPARAÇÃO DE MODELOS:\n")
print(df_comparison.to_string(index=False))
print(f"\n🏆 Melhor modelo: {df_comparison.loc[df_comparison['RMSE'].idxmin(), 'Modelo']}")
print(f"   RMSE: {df_comparison['RMSE'].min():.6f}")

## 📈 Visualizações Comparativas

In [ ]:
# Gráfico de barras comparando RMSE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE
axes[0].bar(df_comparison['Modelo'], df_comparison['RMSE'], color=['blue', 'green', 'red'])
axes[0].set_title('Comparação RMSE', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# MAPE
axes[1].bar(df_comparison['Modelo'], df_comparison['MAPE'], color=['blue', 'green', 'red'])
axes[1].set_title('Comparação MAPE', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MAPE (%)', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparar previsões dos diferentes modelos
fig, axes = plt.subplots(len(all_results), 1, figsize=(14, 4*len(all_results)))

for idx, (config_name, results) in enumerate(all_results.items()):
    ax = axes[idx] if len(all_results) > 1 else axes
    
    forecast_dates = results['forecast_dates']
    predictions = results['predictions']
    labels = results['labels']
    
    # Plotar apenas os últimos 100 pontos para clareza
    n_show = min(100, len(predictions))
    
    ax.plot(forecast_dates[-n_show:], labels[-n_show:], label='Real', color='blue', linewidth=2)
    ax.plot(forecast_dates[-n_show:], predictions[-n_show:], label='Previsão', 
            color='red', linestyle='--', linewidth=2)
    ax.fill_between(forecast_dates[-n_show:], labels[-n_show:], predictions[-n_show:], 
                     alpha=0.2, color='gray')
    
    ax.set_title(f'{config_name} - RMSE: {results["test_metrics"]["RMSE"]:.6f}', 
                fontsize=12, fontweight='bold')
    ax.set_ylabel('Volatilidade', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.xlabel('Data', fontsize=11)
plt.tight_layout()
plt.show()

## 💾 Salvar Modelo Treinado

In [ ]:
# Selecionar o melhor modelo
best_config = df_comparison.loc[df_comparison['RMSE'].idxmin(), 'Modelo']
best_results = all_results[best_config]
best_model = best_results['model']

# Salvar
save_path = f"./models/patchtst_best_{best_config.lower()}"
best_model.save_pretrained(save_path)
best_results['trainer'].save_model(save_path)

print(f"✅ Modelo {best_config} salvo em: {save_path}")
print(f"   Métricas de teste: RMSE={best_results['test_metrics']['RMSE']:.6f}")